<a href="https://colab.research.google.com/github/Ezechinyere87/keep-for-all-ml/blob/main/Ike%26Voke_CS_593A_assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1:** Intent Recognition

**Dataset:** Wizard of Tasks

**Feature Extraction:** TF-IDF Vectorizer

**Model Used:** Five Traditional BaseLine Models and BiLSTM

## **Mount Drive**

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Import all Dependencies**

In [ ]:
# Import all necessary libraries.

import json # Import the json module
import os
import numpy as np
import pandas as pd
import re
from bs4 import BeautifulSoup # BeautifulSoup is a Python library for pulling data out of HTML and XML files.
import nltk # Natural language processing toolkit
import time
from tqdm import tqdm  # for progress tracking
from math import e

import warnings
from bs4 import MarkupResemblesLocatorWarning
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning) # surpress warning

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder


## For BiLSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam



## **Change working directory**

In [ ]:
os.chdir('/content/drive/MyDrive/Fall_2025/CS_593A')

##### This step loads and combines the Wizard of Tasks datasets from the Cooking and DIY domains by reading each JSON file, extracting only the student utterances that have valid intent labels, and returning a Pandas DataFrame with two columns: 'text' for the user utterance and 'intent' for the corresponding label. Combining the two datasets creates a larger, more diverse dataset that enables the model to learn effectively and generalize well to unseen data.

In [ ]:
# Paths to the Wizard of Tasks JSON files (Cooking and DIY domains)
file_path_cooking = '/content/drive/MyDrive/Fall_2025/CS_593A/wizard_of_tasks_cooking_v1.0.json'
file_path_diy = '/content/drive/MyDrive/Fall_2025/CS_593A/wizard_of_tasks_diy_v1.0.json'

# Function: load_wot_json
def load_wot_json(path):
    """
    Load a Wizard of Tasks JSON file and extract student utterances and their intents.

    Args:
        path (str): Path to the JSON dataset file.

    Returns:
        pd.DataFrame: DataFrame containing 'text' and 'intent' columns.
    """

    # Load the JSON data
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Prepare lists to store utterances and intents
    turns = []
    intents = []

    # Iterate through conversations
    for conv_id, conversation in data.items():
        for turn in conversation['turns']:
            # Keep only student turns with a valid intent
            if turn['role'] == 'student' and turn.get('intent') is not None:
                turns.append(turn['text'])
                intents.append(turn['intent'])

    # Return as a DataFrame
    return pd.DataFrame({'text': turns, 'intent': intents})


# Load the Cooking and DIY datasets
df_food = load_wot_json(file_path_cooking)  # Cooking dataset
df_diy = load_wot_json(file_path_diy)       # DIY dataset


# Combine both datasets into one
df_all = pd.concat([df_food, df_diy], ignore_index=True)

# Inspect the combined dataset
display(df_all.head())  # Display first 5 rows
print("\nNumber of samples:", len(df_all))
print("Number of unique intents:", df_all['intent'].nunique())


,text,intent
0,Hi! I love labneh but I've never mixed it with...,ask_question_ingredients_tools
1,After I've chopped all of the herbs and gather...,request_next_step
2,The ingredients are now mixed. What should be ...,request_next_step
3,Once the ingredients are mixed how should I pr...,request_next_step
4,Can I let the ingredients sit for longer to ma...,ask_question_recipe_steps



Number of samples: 9324
Number of unique intents: 6


**Save the Original dataset**

In [ ]:
df_all.to_csv('original_dataset_cooking_&_diy.csv', index=False)

### Read the original dataset into file

In [ ]:
data =pd.read_csv('original_dataset_cooking_&_diy.csv')

display(data.head())
display(data.shape)


,text,intent
0,Hi! I love labneh but I've never mixed it with...,ask_question_ingredients_tools
1,After I've chopped all of the herbs and gather...,request_next_step
2,The ingredients are now mixed. What should be ...,request_next_step
3,Once the ingredients are mixed how should I pr...,request_next_step
4,Can I let the ingredients sit for longer to ma...,ask_question_recipe_steps


(9324, 2)

**Check dataset for the presence of NAN, If present, fill it with an empty string to retain the originality in the dataset without altering and reducing the size of the dataset**

In [ ]:
data.isnull().sum()

,0
text,53
intent,0


In [ ]:
# Fill the missing number in column 'text' with and empty string ('')
data['text'] = data['text'].fillna('')

##**Data Cleaning**

The data cleaning pipeline will follow these steps:

1. Remove HTML tags – Cleans the text by eliminating any embedded HTML elements that do not contribute to the meaning.

2. Replace unwanted characters using regular expressions – Removes all characters except digits, letters, whitespace, and apostrophes to standardize the text.

3. Handle negations – Preserves the meaning of negated expressions (e.g., converting “don’t like” to “do_not_like”) to improve sentiment and intent understanding.

4. Remove stopwords – Eliminates common words (like “the”, “and”, “is”) that carry little semantic value for classification.

5. Convert text to lowercase – Ensures uniformity so that words with different cases are treated identically.

6. Apply stemming – Reduces words to their root forms (e.g., “running” → “run”) to group similar concepts together.

**1. **Remove** HTML**

In [ ]:
def remove_html(text):
    # Remove HTML tags
    bs = BeautifulSoup(text, "html.parser")
    return bs.get_text()

# Apply safely
data['text'] = data['text'].apply(remove_html)

display(data.head())

,text,intent
0,Hi! I love labneh but I've never mixed it with...,ask_question_ingredients_tools
1,After I've chopped all of the herbs and gather...,request_next_step
2,The ingredients are now mixed. What should be ...,request_next_step
3,Once the ingredients are mixed how should I pr...,request_next_step
4,Can I let the ingredients sit for longer to ma...,ask_question_recipe_steps


**2. Replace Characters that are NOT digit, letter, whitespace, or apostrophe with an empty string**



In [ ]:
def clean_text(text):
    text = re.sub(r'[^0-9a-zA-Z\s\']+', '', text).strip()
    return text

# Apply the function to the 'review' column
data['text'] = data['text'].apply(clean_text)

display(data.head())

,text,intent
0,Hi I love labneh but I've never mixed it with ...,ask_question_ingredients_tools
1,After I've chopped all of the herbs and gather...,request_next_step
2,The ingredients are now mixed What should be d...,request_next_step
3,Once the ingredients are mixed how should I pr...,request_next_step
4,Can I let the ingredients sit for longer to ma...,ask_question_recipe_steps


**3. Negation**

In [ ]:
negation_words = [
    'not', "n't", 'never', 'no', 'none', 'nobody', 'nothing', 'nowhere',
    'neither', 'nor', 'hardly', 'barely', 'scarcely', 'without'
]

def handle_negation(text):
    tokens = text.split()
    negated = False
    result = []
    for token in tokens:
        if token in negation_words:
            negated = True
            result.append(token)
            continue
        if negated:
            result.append(f"{token}_NEG")
            if re.search(r'[.!?]', token):  # stop negation at punctuation
                negated = False
        else:
            result.append(token)
    return ' '.join(result)

# Apply negation handling
data['text'] = data['text'].apply(handle_negation)
display(data.head())


,text,intent
0,Hi I love labneh but I've never mixed_NEG it_N...,ask_question_ingredients_tools
1,After I've chopped all of the herbs and gather...,request_next_step
2,The ingredients are now mixed What should be d...,request_next_step
3,Once the ingredients are mixed how should I pr...,request_next_step
4,Can I let the ingredients sit for longer to ma...,ask_question_recipe_steps


**4. StopWord Removal**

In [ ]:
# Stop words removal
# First download the stopwords
nltk.download('stopwords')

english_stop_words = nltk.corpus.stopwords.words('english')
#print(len(english_stop_words))

# Function for stopwords removal
def remove_stop_words(text):
    for stopword in english_stop_words:
        stopword = ' ' + stopword + ' '
        text = text.replace(stopword, ' ')
    return text

# calling the function
data['text'] = data['text'].apply(remove_stop_words)
display(data.head())

# save
#data.to_csv('removed_stopwords_clean_dataset_cooking_&_diy_no_stopwords.csv', index=False)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,text,intent
0,Hi I love labneh I've never mixed_NEG it_NEG w...,ask_question_ingredients_tools
1,After I've chopped herbs gathered ingredients ...,request_next_step
2,The ingredients mixed What done now,request_next_step
3,Once ingredients mixed I proceed,request_next_step
4,Can I let ingredients sit longer make flavors ...,ask_question_recipe_steps


**5. Convert all text to lowercase**

In [ ]:
def convert_to_lowercase(text):
    return text.lower()

# Call the function
data['text'] = data['text'].apply(convert_to_lowercase)

display(data.head())

,text,intent
0,hi i love labneh i've never mixed_neg it_neg w...,ask_question_ingredients_tools
1,after i've chopped herbs gathered ingredients ...,request_next_step
2,the ingredients mixed what done now,request_next_step
3,once ingredients mixed i proceed,request_next_step
4,can i let ingredients sit longer make flavors ...,ask_question_recipe_steps


**6. Stemming**

In [ ]:
# Function for stemming
def text_stemming(text):
    # Ensure the input is a string
    #text = str(text)
    stemmer = nltk.porter.PorterStemmer()
    stemmed = ' '.join([stemmer.stem(token) for token in text.split()])
    return stemmed

# calling the stemming function
data['text'] = data['text'].apply(text_stemming)

display(data.head())

,text,intent
0,hi i love labneh i'v never mixed_neg it_neg wi...,ask_question_ingredients_tools
1,after i'v chop herb gather ingredi i mix altogeth,request_next_step
2,the ingredi mix what done now,request_next_step
3,onc ingredi mix i proceed,request_next_step
4,can i let ingredi sit longer make flavor stronger,ask_question_recipe_steps


**Save the Clean dataset for further use**

In [ ]:
data.to_csv('complete_clean_dataset_cooking_&_diy.csv', index=False)

##**Feature Engineering**

In [ ]:
# Read data
df = pd.read_csv('complete_clean_dataset_cooking_&_diy.csv')
display(df.head())

,text,intent
0,hi i love labneh i'v never mixed_neg it_neg wi...,ask_question_ingredients_tools
1,after i'v chop herb gather ingredi i mix altogeth,request_next_step
2,the ingredi mix what done now,request_next_step
3,onc ingredi mix i proceed,request_next_step
4,can i let ingredi sit longer make flavor stronger,ask_question_recipe_steps


### **Compute the TF-IDF features ALL the ngram_range, then split the dataset into 80% training, 10% validation, and 10% test sets, and display the shapes of each split**

Note: The 10% validation set is used to evaluate how the trained model performs during development. It helps in fine-tuning or adjusting the model’s parameters if necessary, while keeping the test set untouched to ensure an unbiased final evaluation and prevent the model from learning its features.


In [ ]:
uni_vectorizer = TfidfVectorizer(ngram_range=(1, 1))
uni_tfidf = uni_vectorizer.fit_transform(df['text'].fillna(''))
y = df['intent']

# Split data into 80% train, 10% validation and 10% test set
X_train_uni, X_temp_uni, y_train_uni, y_temp_uni = train_test_split(uni_tfidf, y, test_size=0.2, random_state=42, stratify=y)
X_val_uni, X_test_uni, y_val_uni, y_test_uni = train_test_split(X_temp_uni, y_temp_uni, test_size=0.5, random_state=42, stratify=y_temp_uni)

print('\n-------Below are the shapes of Unigram Features------------\n')
print(f'Xtrain_uni: {X_train_uni.shape}; X_val_uni: {X_val_uni.shape}; X_test_uni: {X_test_uni.shape};')
print(f'y_train_uni: {y_train_uni.shape}; y_val_uni: {y_val_uni.shape}; y_test_uni: {y_test_uni.shape};')

#-------Unigram + Bigrams--------------
bi_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
bi_tfidf = bi_vectorizer.fit_transform(df['text'].fillna(''))

# Split data into 80% train, 10% validation and 10% test set
X_train_bi, X_temp_bi, y_train_bi, y_temp_bi = train_test_split(bi_tfidf, y, test_size=0.2, random_state=42, stratify=y)
X_val_bi, X_test_bi, y_val_bi, y_test_bi = train_test_split(X_temp_bi, y_temp_bi, test_size=0.5, random_state=42, stratify=y_temp_bi)

print('\n-------Below are the shapes of Unigram + Bigrams Features------------\n')
print(f'X_train: {X_train_bi.shape}; X_val: {X_val_bi.shape}; X_test: {X_test_bi.shape};')
print(f'y_train: {y_train_bi.shape}; y_val: {y_val_bi.shape}; y_test: {y_test_bi.shape};')


#--------------Unigram + Bigrams + Trigrams-----------------
tri_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tri_tfidf = tri_vectorizer.fit_transform(df['text'].fillna(''))

# Split data into 80% train, 10% validation and 10% test set
X_train_tri, X_temp_tri, y_train_tri, y_temp_tri = train_test_split(bi_tfidf, y, test_size=0.2, random_state=42, stratify=y)
X_val_tri, X_test_tri, y_val_tri, y_test_tri = train_test_split(X_temp_tri, y_temp_tri, test_size=0.5, random_state=42, stratify=y_temp_tri)

print('\n-------Below are the shapes of Unigram + Bigrams + Trigrams Features------------\n')
print(f'X_train: {X_train_tri.shape}; X_val: {X_val_tri.shape}; X_test: {X_test_tri.shape};')
print(f'y_train: {y_train_tri.shape}; y_val: {y_val_tri.shape}; y_test: {y_test_tri.shape};')


-------Below are the shapes of Unigram Features------------

Xtrain_uni: (7459, 4230); X_val_uni: (932, 4230); X_test_uni: (933, 4230);
y_train_uni: (7459,); y_val_uni: (932,); y_test_uni: (933,);

-------Below are the shapes of Unigram + Bigrams Features------------

X_train: (7459, 35208); X_val: (932, 35208); X_test: (933, 35208);
y_train: (7459,); y_val: (932,); y_test: (933,);

-------Below are the shapes of Unigram + Bigrams + Trigrams Features------------

X_train: (7459, 35208); X_val: (932, 35208); X_test: (933, 35208);
y_train: (7459,); y_val: (932,); y_test: (933,);


## **Applyiny Traditional ML classifiers**




###### The following traditional machine learning baseline models were applied: Naive Bayes, Logistic Regression, Random Forest, XGBoost Classifier, and Linear SVC.

| **Model**           | **Reason for Selection**                                                                  |
| ------------------- | ----------------------------------------------------------------------------------------- |
| **Naive Bayes**         | Simple, fast, and effective for text classification with TF-IDF features.                 |
| **Logistic Regression** | Strong linear baseline with interpretable feature weights.                                |
| **Random Forest**       | Captures non-linear relationships and reduces overfitting through ensemble averaging.     |
| **XGBoost Classifier**  | Powerful gradient boosting method that often improves performance on structured features. |
| **Linear SVC**         | Efficient linear classifier suitable for high-dimensional sparse TF-IDF vectors.          |






###### Define a function that takes train, validation and test imput as parameters to train and evaluate multiple machine learning models on a given dataset that has already been split into training, validation, and test sets

## Due to class imbalance, we used Weighted average insteda of Macro to evaluate the whole performance


In [ ]:
# Model Training Function
def model_training(X_train, y_train, X_val, y_val, X_test, y_test):
    # Encode string labels to numerical labels
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
    y_test_encoded = label_encoder.transform(y_test)

    # Define models with tuned hyperparameters
    models = {
        'MultinomialNB': MultinomialNB(alpha=1.0, fit_prior=True),
        'LogisticRegression': LogisticRegression(
            C=1.0, max_iter=5000, random_state=42, solver='saga',
            penalty='l2', tol=0.0001, class_weight=None),
        'RandomForestClassifier': RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_split=2,
            min_samples_leaf=1, max_features='sqrt', random_state=42,
            criterion='entropy', n_jobs=-1),
        'XGBClassifier': XGBClassifier(
            n_estimators=300, learning_rate=0.1, max_depth=6, subsample=0.8,
            colsample_bytree=0.8, gamma=0, random_state=42, eval_metric='mlogloss', n_jobs=-1),
        'LinearSVC': LinearSVC(
            C=1.0, loss='squared_hinge', max_iter=5000, class_weight=None, random_state=42)
    }

    results = []

    for name, model in tqdm(models.items(), desc='Training Models'):
        start_time = time.time()
        model.fit(X_train, y_train_encoded)
        train_time = time.time() - start_time

        # Evaluation on Validation Set
        y_val_pred = label_encoder.inverse_transform(model.predict(X_val))
        acc_val = accuracy_score(y_val, y_val_pred)
        prec_val = precision_score(y_val, y_val_pred, average='weighted', zero_division=0)
        rec_val = recall_score(y_val, y_val_pred, average='weighted')
        f1_val = f1_score(y_val, y_val_pred, average='weighted')

        # Evaluation on Test Set
        y_test_pred = label_encoder.inverse_transform(model.predict(X_test))
        acc_test = accuracy_score(y_test, y_test_pred)
        prec_test = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
        rec_test = recall_score(y_test, y_test_pred, average='weighted')
        f1_test = f1_score(y_test, y_test_pred, average='weighted')

        results.append([
            name,
            round(acc_val, 4), round(prec_val, 4), round(rec_val, 4), round(f1_val, 4),
            round(acc_test, 4), round(prec_test, 4), round(rec_test, 4), round(f1_test, 4),
            round(train_time, 4)
        ])

    columns = [
        'Model',
        'Accuracy_val', 'Precision_val', 'Recall_val', 'F1_val',
        'Accuracy_test', 'Precision_test', 'Recall_test', 'F1_test',
        'Train_Time(s)'
    ]

    comparison_df = pd.DataFrame(results, columns=columns)
    comparison_df.sort_values(by='F1_val', ascending=False, inplace=True)
    comparison_df.reset_index(drop=True, inplace=True)

    return comparison_df


###### **Evaluation Result for Unigram**

In [ ]:
# Call the model_training function with the prepared data
model_comparison = model_training(X_train_uni, y_train_uni, X_val_uni, y_val_uni, X_test_uni, y_test_uni)

# Display the comparison DataFrame
print('\n---------Model Comparison (Unigram)---------\n')
model_comparison



Training Models: 100%|██████████| 5/5 [01:39<00:00, 19.87s/it]


---------Model Comparison (Unigram)---------



,Model,Accuracy_val,Precision_val,Recall_val,F1_val,Accuracy_test,Precision_test,Recall_test,F1_test,Train_Time(s)
0,XGBClassifier,0.8240,0.8237,0.8240,0.8170,0.8156,0.8088,0.8156,0.8085,55.3939
1,RandomForestClassifier,0.8230,0.8268,0.8230,0.8135,0.8274,0.8189,0.8274,0.8177,42.8142
2,LinearSVC,0.8047,0.7951,0.8047,0.7955,0.7942,0.7828,0.7942,0.7873,0.0895
3,LogisticRegression,0.8026,0.7983,0.8026,0.7900,0.8049,0.7969,0.8049,0.7957,0.1612
4,MultinomialNB,0.7371,0.7547,0.7371,0.7032,0.7149,0.7193,0.7149,0.6746,0.0059


###### **Evaluation Result for Unigram + Bigrams**

In [ ]:
# Call the model_training function with the prepared data
model_comparison = model_training(X_train_bi, y_train_bi, X_val_bi, y_val_bi, X_test_bi, y_test_bi)

# Display the comparison DataFrame
print('\n---------Model Comparison (Unigram + Bigrams)---------\n')
model_comparison

Training Models: 100%|██████████| 5/5 [05:13<00:00, 62.66s/it]



---------Model Comparison (Unigram + Bigrams)---------



,Model,Accuracy_val,Precision_val,Recall_val,F1_val,Accuracy_test,Precision_test,Recall_test,F1_test,Train_Time(s)
0,RandomForestClassifier,0.8326,0.8395,0.8326,0.8240,0.8307,0.8237,0.8307,0.8198,160.1788
1,XGBClassifier,0.8273,0.8271,0.8273,0.8204,0.8103,0.8042,0.8103,0.8031,150.9814
2,LinearSVC,0.8251,0.8283,0.8251,0.8172,0.8349,0.8274,0.8349,0.8268,0.1273
3,LogisticRegression,0.8240,0.8262,0.8240,0.8121,0.8253,0.8206,0.8253,0.8143,0.3646
4,MultinomialNB,0.7221,0.7565,0.7221,0.6864,0.7170,0.7496,0.7170,0.6804,0.0167


###### **Evaluation Result for Unigram + Bigrams + Trigrams**

In [ ]:
# Call the model_training function with the prepared data
model_comparison = model_training(X_train_tri, y_train_tri, X_val_tri, y_val_tri, X_test_tri, y_test_tri)

# Display the comparison DataFrame
print('\n---------Model Comparison (Unigram + Bigrams)---------\n')
model_comparison

Training Models: 100%|██████████| 5/5 [04:22<00:00, 52.51s/it]


---------Model Comparison (Unigram + Bigrams)---------



,Model,Accuracy_val,Precision_val,Recall_val,F1_val,Accuracy_test,Precision_test,Recall_test,F1_test,Train_Time(s)
0,RandomForestClassifier,0.8326,0.8395,0.8326,0.8240,0.8307,0.8237,0.8307,0.8198,150.8064
1,XGBClassifier,0.8273,0.8271,0.8273,0.8204,0.8103,0.8042,0.8103,0.8031,110.5440
2,LinearSVC,0.8251,0.8283,0.8251,0.8172,0.8349,0.8274,0.8349,0.8268,0.1224
3,LogisticRegression,0.8240,0.8262,0.8240,0.8121,0.8253,0.8206,0.8253,0.8143,0.2118
4,MultinomialNB,0.7221,0.7565,0.7221,0.6864,0.7170,0.7496,0.7170,0.6804,0.0086


####**Summary Table For all models across the three n-gram TF-IDF**


| Model                | TF-IDF            | F1_val | F1_test | Train_Time(s) |
| -------------------- | ----------------- | ------ | ------- | ------------- |
| LinearSVC            | Uni + Bi          | 0.8172 | 0.8268  | 0.127         |
| LinearSVC            | Uni + Bi + Tri    | 0.8172 | 0.8268  | 0.122         |
| RandomForest         | Uni + Bi          | 0.8240 | 0.8198  | 160.18        |
| RandomForest         | Uni + Bi + Tri    | 0.8240 | 0.8198  | 150.81        |
| RandomForest         | Unigram           | 0.8135 | 0.8177  | 42.81         |
| LogisticRegression   | Uni + Bi          | 0.8121 | 0.8143  | 0.365         |
| LogisticRegression   | Uni + Bi + Tri    | 0.8121 | 0.8143  | 0.212         |
| XGBClassifier        | Unigram           | 0.8170 | 0.8085  | 55.39         |
| XGBClassifier        | Uni + Bi          | 0.8204 | 0.8031  | 150.98        |
| XGBClassifier        | Uni + Bi + Tri    | 0.8204 | 0.8031  | 110.54        |
| LogisticRegression   | Unigram           | 0.7900 | 0.7957  | 0.161         |
| LinearSVC            | Unigram           | 0.7955 | 0.7873  | 0.089         |
| MultinomialNB        | Unigram           | 0.7032 | 0.6746  | 0.006         |
| MultinomialNB        | Uni + Bi          | 0.6864 | 0.6804  | 0.017         |
| MultinomialNB        | Uni + Bi + Tri    | 0.6864 | 0.6804  | 0.009         |



**Notes**

**Top performers:**
*   LinearSVC with unigram + bigram/trigram — best F1_test = 0.8268
*   RandomForest comes next (F1_test ≈ 0.8198)

**Middle tier:**

*   XGBClassifier with unigram/bigrams/trigrams (F1_test ≈ 0.8031–0.8085)

**Lowest performers:**

*    MultinomialNB consistently at the bottom (F1_test ≈ 0.68–0.67)

**Efficiency:**

*   LinearSVC is fastest among top performers

*   Tree-based models (RandomForest, XGB) take much longer to train


#####**BiLSTM Text Classification**

This was used because of its ability to capture contextual information from both directions in a sequence, learn long-term dependencies, and reduce manual feature engineering, making it especially effective for nuanced language tasks.

In [ ]:
# ==============================
# Step 0: Text Cleaning Functions
# ==============================
def remove_html(text):
    # Ensure input is a string before passing to BeautifulSoup
    text = str(text) if text is not None else ''
    # Remove HTML tags
    bs = BeautifulSoup(text, "html.parser")
    return bs.get_text()

# Function to convert text to lowercase
def convert_to_lowercase(text):
    return text.lower()

# ==============================
# Step 1: Load Datasets
# ==============================
df = pd.read_csv('original_dataset_cooking_&_diy.csv')


# Fill missing values with empty strings before applying cleaning functions
df['text'] = df['text'].fillna('')

# Apply cleaning functions to dataset
df['text'] = df['text'].apply(remove_html)
df['text'] = df['text'].apply(convert_to_lowercase)


# ==============================
# Step 2: Hyperparameters
# ==============================
MAX_VOCAB = 10000
MAX_LEN = 30
EMBED_DIM = 100

# ==============================
# Step 3: Encode labels
# ==============================
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['intent'])
num_classes = len(set(y_encoded))

# ==============================
# Step 4: Train-test split
# ==============================
X_train_seq, X_test_seq, y_train_enc, y_test_enc = train_test_split(
    df['text'], y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

X_train_seq = X_train_seq.fillna('').astype(str)
X_test_seq = X_test_seq.fillna('').astype(str)

# ==============================
# Step 5: Tokenization & Padding
# ==============================
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_seq)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_seq), maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test_seq), maxlen=MAX_LEN, padding='post')

y_train_cat = to_categorical(y_train_enc, num_classes=num_classes)
y_test_cat = to_categorical(y_test_enc, num_classes=num_classes)

# ==============================
# Step 6: Build BiLSTM Model
# ==============================
model_bilstm = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=EMBED_DIM),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model_bilstm.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

# ==============================
# Step 7: Train BiLSTM
# ==============================
start_time = time.time()
history = model_bilstm.fit(
    X_train_pad, y_train_cat,
    validation_split=0.1,
    epochs=15,
    batch_size=32,
    verbose=2
)
train_time = time.time() - start_time

# ==============================
# Step 8: Evaluate BiLSTM
# ==============================
# Validation metrics
val_size = int(0.1 * len(X_train_pad))
X_val_pad = X_train_pad[-val_size:]
y_val_true = np.argmax(y_train_cat[-val_size:], axis=1)
y_val_pred = np.argmax(model_bilstm.predict(X_val_pad), axis=1)
acc_val = accuracy_score(y_val_true, y_val_pred)
f1_val = f1_score(y_val_true, y_val_pred, average='macro')

# Test metrics
y_test_pred = np.argmax(model_bilstm.predict(X_test_pad), axis=1)
acc_test = accuracy_score(y_test_enc, y_test_pred)
f1_test = f1_score(y_test_enc, y_test_pred, average='macro')

# ==============================
# Step 9: Store Results
# ==============================
results_df = pd.DataFrame([{
    'Model': 'BiLSTM',
    'Accuracy_val': round(acc_val, 4),
    'F1_val': round(f1_val, 4),
    'Accuracy_test': round(acc_test, 4),
    'F1_test': round(f1_test, 4),
    'Train_Time(s)': round(train_time, 4)
}])
print(results_df)
print("\nClassification Report (Test Set):")
print(classification_report(y_test_enc, y_test_pred, target_names=label_encoder.classes_, zero_division=0))

Epoch 1/15
210/210 - 16s - 78ms/step - accuracy: 0.6686 - loss: 0.8917 - val_accuracy: 0.8324 - val_loss: 0.5749
Epoch 2/15
210/210 - 10s - 47ms/step - accuracy: 0.8378 - loss: 0.4948 - val_accuracy: 0.8485 - val_loss: 0.4887
Epoch 3/15
210/210 - 8s - 39ms/step - accuracy: 0.8865 - loss: 0.3752 - val_accuracy: 0.8539 - val_loss: 0.4898
Epoch 4/15
210/210 - 11s - 54ms/step - accuracy: 0.9130 - loss: 0.2974 - val_accuracy: 0.8432 - val_loss: 0.5529
Epoch 5/15
210/210 - 10s - 47ms/step - accuracy: 0.9340 - loss: 0.2389 - val_accuracy: 0.8231 - val_loss: 0.6135
Epoch 6/15
210/210 - 9s - 43ms/step - accuracy: 0.9409 - loss: 0.2038 - val_accuracy: 0.8472 - val_loss: 0.6027
Epoch 7/15
210/210 - 9s - 45ms/step - accuracy: 0.9479 - loss: 0.1762 - val_accuracy: 0.8070 - val_loss: 0.7290
Epoch 8/15
210/210 - 11s - 50ms/step - accuracy: 0.9525 - loss: 0.1646 - val_accuracy: 0.8271 - val_loss: 0.8056
Epoch 9/15
210/210 - 9s - 42ms/step - accuracy: 0.9581 - loss: 0.1409 - val_accuracy: 0.8150 - val_

## **The BiLSTM result shows a weighted-average accuracy of 0.80 on the validation set and 0.79 on the test set, which is comparable to the performance observed with the traditional baseline models.**